In [1]:
#IMPORTACION DE LIBERIAS

# Manejo de datos
import numpy as np
import pandas as pd
from IPython.display import display

from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

In [2]:
# Cargar las credenciales del archivo .env
load_dotenv()

USER = os.getenv('POSTGRES_USER')
PASSWORD = os.getenv('POSTGRES_PASSWORD')
HOST = os.getenv('POSTGRES_HOST')
PORT = os.getenv('POSTGRES_PORT') 
DB = os.getenv('POSTGRES_DB')

# 2. Crear la conexión a PostgreSQL
engine = create_engine(f'postgresql://{USER}:{PASSWORD}@{HOST}:{PORT}/{DB}')

print("Conexión a la base de datos establecida")

Conexión a la base de datos establecida


In [3]:
# ORDERS
# 1. Definir la consulta SQL
query = """
SELECT * FROM olist_orders_dataset 
"""

# 2. Ejecutar la consulta y guardarla
df_orders = pd.read_sql(query, engine)

In [4]:
# ORDER ITEMS
query = """
SELECT * FROM olist_order_items_dataset
"""
df_order_items = pd.read_sql(query, engine)

In [5]:
# PRODUCTS
query = """
SELECT * FROM olist_products_dataset
"""
df_products = pd.read_sql(query, engine)


In [10]:
# Funcion de analisis exploratorio base
def exploracion_profesional(df):
    print(" ANÁLISIS EXPLORATORIO BASE\n")
    print("Informacion general: ")
    
    df.info()
    
    print()
    print(f"Filas duplicadas totales: {df.duplicated().sum()}\n")
    
    print("Primeras 5 filas:")
    print(df.head())
    print()
    
    print("Descripción Estadística (Variables Numéricas): ")
    print(df.describe().round(2))
    print()
    
    print("Resumen de Nulos y Valores Únicos por Columna:")
    
    resumen = pd.DataFrame({
        'Tipo de Dato': df.dtypes,
        'Valores Nulos': df.isnull().sum(),
        '% Nulos': (df.isnull().sum() / len(df) * 100).round(2),
        'Valores Únicos': df.nunique()
    })
    
    display(resumen)
    
    print("\n Frecuencia de categorías (Solo columnas con menos de 40 valores únicos):")
    
    for column in df.columns:
        
        print(f"Total de valores unicos: {df[column].nunique()} ")
        
        if df[column].nunique() < 40:
            print(f"\n- Columna: '{column}'")
            print(df[column].value_counts())

In [11]:
exploracion_profesional(df_orders)

 ANÁLISIS EXPLORATORIO BASE

Informacion general: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB

Filas duplicadas totales: 0

Primeras 5 filas:
                           order_id                       customer_id  \
0  e481f51cbdc54678b7cc49136f2d6af7  9ef432eb6251297304e76186b10a928d   
1  53cdb2fc8bc7dc

,Tipo de Dato,Valores Nulos,% Nulos,Valores Únicos
order_id,object,0,0.00,99441
customer_id,object,0,0.00,99441
order_status,object,0,0.00,8
order_purchase_timestamp,object,0,0.00,98875
order_approved_at,object,160,0.16,90733
order_delivered_carrier_date,object,1783,1.79,81018
order_delivered_customer_date,object,2965,2.98,95664
order_estimated_delivery_date,object,0,0.00,459



 Frecuencia de categorías (Solo columnas con menos de 40 valores únicos):
Total de valores unicos: 99441 
Total de valores unicos: 99441 
Total de valores unicos: 8 

- Columna: 'order_status'
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64
Total de valores unicos: 98875 
Total de valores unicos: 90733 
Total de valores unicos: 81018 
Total de valores unicos: 95664 
Total de valores unicos: 459 


# Observaciones olist_orders_dataset:
- Todas las columnas son de tipo object / Columnas con posible cambio a datetime para aplicar data engineering (temporalidad)
- order_status tiene 8 posibles valores diferentes, delivered es casi la mayoria absoluta, observar a fondo los casos anomalos
- order_estimated_delivery_date a diferencia de otras columnas de tiempo, tiene menos de 500 valores unicos, revisar correlacion del porque hay una fecha de coincidencia
- Antes de entrenar es necesario quitar filas que puedan introducir el data leakage
- Los nulos no son errores del sistema; están vacíos simplemente porque esos pedidos nunca llegaron al cliente.

In [8]:
exploracion_profesional(df_order_items)

 ANÁLISIS EXPLORATORIO BASE

Informacion general: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB

Filas duplicadas totales: 0

Primeras 5 filas:
                           order_id  order_item_id  \
0  00010242fe8c5a6d1ba2dd792cb16214              1   
1  00018f77f2f0320c557190d7a144bdd3              1   
2  000229ec398224ef6ca0657da4fc703e              1   
3  00024acbcdf0a6daa1e931b038114c75              1   


,Tipo de Dato,Valores Nulos,% Nulos,Valores Únicos
order_id,object,0,0.0,98666
order_item_id,int64,0,0.0,21
product_id,object,0,0.0,32951
seller_id,object,0,0.0,3095
shipping_limit_date,object,0,0.0,93318
price,float64,0,0.0,5968
freight_value,float64,0,0.0,6999



 Frecuencia de categorías (Solo columnas con menos de 40 valores únicos):
Total de valores unicos: 98666 
Total de valores unicos: 21 

- Columna: 'order_item_id'
order_item_id
1     98666
2      9803
3      2287
4       965
5       460
6       256
7        58
8        36
9        28
10       25
11       17
12       13
13        8
14        7
15        5
16        3
17        3
18        3
19        3
20        3
21        1
Name: count, dtype: int64
Total de valores unicos: 32951 
Total de valores unicos: 3095 
Total de valores unicos: 93318 
Total de valores unicos: 5968 
Total de valores unicos: 6999 


# Observaciones olist_order_items_dataset:

In [9]:
exploracion_profesional(df_products)

 ANÁLISIS EXPLORATORIO BASE

Informacion general: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  object 
 1   product_category_name       32341 non-null  object 
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), object(2)
memory usage: 2.3+ MB

Filas duplicadas totales: 0

Primeras 5 filas:
                         product_id  product_category_name  \
0  1e9e8ef04dbcff4541ed26657ea517e5             perf

,Tipo de Dato,Valores Nulos,% Nulos,Valores Únicos
product_id,object,0,0.00,32951
product_category_name,object,610,1.85,73
product_name_lenght,float64,610,1.85,66
product_description_lenght,float64,610,1.85,2960
product_photos_qty,float64,610,1.85,19
product_weight_g,float64,2,0.01,2204
product_length_cm,float64,2,0.01,99
product_height_cm,float64,2,0.01,102
product_width_cm,float64,2,0.01,95



 Frecuencia de categorías (Solo columnas con menos de 40 valores únicos):
Total de valores unicos: 32951 
Total de valores unicos: 73 
Total de valores unicos: 66 
Total de valores unicos: 2960 
Total de valores unicos: 19 

- Columna: 'product_photos_qty'
product_photos_qty
1.0     16489
2.0      6263
3.0      3860
4.0      2428
5.0      1484
6.0       968
7.0       343
8.0       192
9.0       105
10.0       95
11.0       46
12.0       35
13.0        9
15.0        8
17.0        7
14.0        5
18.0        2
20.0        1
19.0        1
Name: count, dtype: int64
Total de valores unicos: 2204 
Total de valores unicos: 99 
Total de valores unicos: 102 
Total de valores unicos: 95 


# Observaciones olist_products_dataset:
presencia de nulos en la mayoria de columnas(Id es la excepcion)
